# OCR Pipeline Evaluation - Pretrained Models Baseline

This notebook evaluates 3 OCR pipelines using **pretrained/base models** on our custom dataset:

1. **EasyOCR + TrOCR + Qwen2-VL** 
2. **PaddleOCR + Qwen2-VL**
3. **DBNet + Parseq + Qwen2-VL**

**Output Metrics**: Accuracy, ANLS, Semantic Score, CER, WER, FPS

## 1. Setup & Install Dependencies

In [1]:
# Install all required packages
!pip install -q easyocr transformers torch torchvision pillow opencv-python-headless
!pip install -q paddlepaddle paddleocr
!pip install -q rapidfuzz jiwer tqdm
!pip install -q qwen-vl-utils accelerate
!pip install -q timm groq python-dotenv

print("✅ All packages installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.7/193.7 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 

In [2]:
import os
import json
import time
import re
import numpy as np
import cv2
import torch
from PIL import Image, ImageFilter
from tqdm import tqdm
from pathlib import Path

# ============================================================
# GROQ API KEYS - Add multiple keys for fallback!
# When one key runs out, it automatically switches to the next
# ============================================================
GROQ_API_KEYS = [
    "YOUR_GROQ_API_KEY_1",  # Primary key
    "YOUR_GROQ_API_KEY_2",  # Backup key 1
    "YOUR_GROQ_API_KEY_3",  # Backup key 2
    # Add more keys as needed...
]

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Loaded {len([k for k in GROQ_API_KEYS if not k.startswith('YOUR')])} valid Groq API keys")

Device: cpu
Loaded 0 valid Groq API keys


## 2. Load Dataset

In [3]:
# Dataset paths - adjust based on your Kaggle setup
DATA_PATH = "/kaggle/input/dronevqa-data/DATA"
ANNO_PATH = "/kaggle/input/dronevqa-data/textvqa_dataset.json"

# Load annotations
with open(ANNO_PATH, 'r') as f:
    dataset = json.load(f)
samples = dataset['data']
print(f"📋 Loaded {len(samples)} samples with questions/answers")

# Blur level (same as fine-tuned testing)
BLUR_RADIUS = 5.0
LIMIT = 100  # Number of samples to evaluate

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/dronevqa-data/textvqa_dataset.json'

## 3. Define ALL Metrics (Accuracy, ANLS, Semantic, CER, WER)

In [ ]:
from rapidfuzz import fuzz
import jiwer

def calculate_anls(pred: str, gt_list: list, threshold: float = 0.5) -> float:
    """Average Normalized Levenshtein Similarity"""
    pred = pred.lower().strip()
    if not pred and not gt_list:
        return 1.0
    best_score = 0.0
    for gt in gt_list:
        gt = gt.lower().strip()
        if not gt:
            continue
        similarity = fuzz.ratio(pred, gt) / 100.0
        if similarity < threshold:
            similarity = 0.0
        best_score = max(best_score, similarity)
    return best_score

def calculate_accuracy(pred: str, gt_list: list) -> float:
    """Exact match accuracy"""
    pred = pred.lower().strip()
    for gt in gt_list:
        if gt.lower().strip() == pred:
            return 1.0
    return 0.0

def calculate_cer_wer(pred: str, gt_list: list) -> tuple:
    """Character Error Rate and Word Error Rate"""
    if not gt_list:
        return 1.0, 1.0
    best_cer, best_wer = float('inf'), float('inf')
    pred = pred.lower().strip()
    for gt in gt_list:
        gt = gt.lower().strip()
        if not gt and not pred:
            c, w = 0.0, 0.0
        elif not gt:
            c, w = 1.0, 1.0
        else:
            c = jiwer.cer(gt, pred)
            w = jiwer.wer(gt, pred)
        best_cer, best_wer = min(best_cer, c), min(best_wer, w)
    return best_cer, best_wer


# ============================================================
# SEMANTIC JUDGE WITH MULTI-KEY FALLBACK
# Automatically switches to next key when rate limit is hit
# ============================================================
class SemanticJudge:
    def __init__(self, api_keys: list):
        from groq import Groq
        self.Groq = Groq
        
        # Filter out placeholder keys
        self.api_keys = [k for k in api_keys if k and not k.startswith('YOUR')]
        self.current_key_index = 0
        self.clients = []
        self.failed_keys = set()
        
        # Initialize clients for all keys
        for key in self.api_keys:
            try:
                self.clients.append(Groq(api_key=key))
            except:
                pass
        
        if self.clients:
            print(f"[SemanticJudge] Initialized with {len(self.clients)} Groq API keys")
            print(f"[SemanticJudge] Will auto-switch if a key runs out!")
        else:
            print("[SemanticJudge] ❌ No valid API keys! Add keys to GROQ_API_KEYS list.")
            raise ValueError("No valid Groq API keys provided!")
    
    def _get_current_client(self):
        """Get current working client, skip failed ones"""
        while self.current_key_index < len(self.clients):
            if self.current_key_index not in self.failed_keys:
                return self.clients[self.current_key_index]
            self.current_key_index += 1
        return None
    
    def _switch_to_next_key(self):
        """Mark current key as failed and switch to next"""
        self.failed_keys.add(self.current_key_index)
        self.current_key_index += 1
        if self.current_key_index < len(self.clients):
            print(f"\n⚠️ API key {self.current_key_index} exhausted. Switching to key {self.current_key_index + 1}...")
            return True
        else:
            print("\n❌ ALL API KEYS EXHAUSTED! Cannot continue semantic scoring.")
            return False
    
    def judge(self, question: str, gt_list: list, prediction: str) -> float:
        """Judge semantic correctness using Groq with auto key-switching"""
        # Quick exact match check
        pred_clean = prediction.lower().strip()
        for gt in gt_list:
            if gt.lower().strip() == pred_clean:
                return 1.0
        
        # Try with Groq
        max_retries = len(self.clients)
        
        for attempt in range(max_retries):
            client = self._get_current_client()
            if not client:
                raise RuntimeError("All Groq API keys exhausted!")
            
            try:
                gts_str = ", ".join([f'"{g}"' for g in gt_list[:5]])
                prompt = f"""You are a strict judge for VQA.
Question: {question}
Ground Truth: [{gts_str}]
Prediction: "{prediction}"
Reply ONLY with JSON: {{"correct": true}} or {{"correct": false}}"""
                
                response = client.chat.completions.create(
                    model="llama-3.1-8b-instant",
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.0,
                    max_tokens=50
                )
                
                content = response.choices[0].message.content.lower()
                return 1.0 if '"correct": true' in content or '"correct":true' in content else 0.0
                
            except Exception as e:
                error_str = str(e).lower()
                # Check if it's a rate limit error
                if 'rate' in error_str or 'limit' in error_str or '429' in error_str or 'quota' in error_str:
                    if not self._switch_to_next_key():
                        raise RuntimeError("All Groq API keys exhausted!")
                else:
                    # Other error, still try next key
                    print(f"\n⚠️ Groq error: {e}")
                    if not self._switch_to_next_key():
                        raise RuntimeError("All Groq API keys exhausted!")
        
        raise RuntimeError("All Groq API keys exhausted!")


# Initialize the judge with all keys
judge = SemanticJudge(GROQ_API_KEYS)
print("✅ All metrics defined (Accuracy, ANLS, Semantic, CER, WER)")

## 4. Load Qwen2-VL for VQA

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoTokenizer, AutoProcessor

print("Loading Qwen2-VL-2B...")
qwen_model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)
qwen_processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")
print("✅ Qwen2-VL loaded")

def ask_qwen(image_path: str, question: str, ocr_text: str) -> str:
    """Ask Qwen VLM a question with OCR context"""
    prompt = f"""OCR Text detected in image: {ocr_text}

Question: {question}

Instructions: Answer ONLY with the answer text. Limit to 1-5 words."""
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": prompt}
            ]
        }
    ]
    
    text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image = Image.open(image_path).convert("RGB")
    inputs = qwen_processor(text=[text], images=[image], padding=True, return_tensors="pt").to(device)
    
    with torch.no_grad():
        output_ids = qwen_model.generate(**inputs, max_new_tokens=50)
    
    output = qwen_processor.batch_decode(output_ids, skip_special_tokens=True)[0]
    # Extract just the answer
    answer = output.split("assistant")[-1].strip() if "assistant" in output else output
    return answer.strip()

## 5. Load OCR Models

In [ ]:
import easyocr
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from paddleocr import PaddleOCR
from torchvision import transforms as T

# ========== Pipeline 1: EasyOCR + TrOCR ==========
print("Loading EasyOCR + TrOCR...")
easyocr_reader = easyocr.Reader(['en'], gpu=True)
trocr_processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-str")
trocr_model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-str").to(device).eval()

def run_easyocr_trocr(image_rgb: np.ndarray) -> str:
    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    results = easyocr_reader.readtext(image_bgr, detail=1)
    h, w = image_bgr.shape[:2]
    ocr_texts = []
    for (bbox, _, conf) in results:
        pts = np.array(bbox, dtype=np.int32)
        x_min, x_max = max(0, np.min(pts[:, 0]) - 5), min(w, np.max(pts[:, 0]) + 5)
        y_min, y_max = max(0, np.min(pts[:, 1]) - 5), min(h, np.max(pts[:, 1]) + 5)
        if (x_max - x_min) >= 10 and (y_max - y_min) >= 10:
            crop = image_rgb[y_min:y_max, x_min:x_max]
            try:
                pil_img = Image.fromarray(crop)
                pixel_values = trocr_processor(images=pil_img, return_tensors="pt").pixel_values.to(device)
                with torch.no_grad():
                    ids = trocr_model.generate(pixel_values, max_length=64)
                text = trocr_processor.batch_decode(ids, skip_special_tokens=True)[0]
                if text:
                    ocr_texts.append(text)
            except:
                pass
    return " ".join(ocr_texts)

# ========== Pipeline 2: PaddleOCR ==========
print("Loading PaddleOCR...")
paddle_ocr = PaddleOCR(use_angle_cls=True, lang='en', use_gpu=True, show_log=False)

def run_paddleocr(image_rgb: np.ndarray) -> str:
    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    result = paddle_ocr.ocr(image_bgr, cls=True)
    ocr_texts = []
    if result and result[0]:
        for line in result[0]:
            text, conf = line[1][0], line[1][1]
            if conf > 0.5:
                ocr_texts.append(text)
    return " ".join(ocr_texts)

# ========== Pipeline 3: DBNet + Parseq ==========
print("Loading DBNet + Parseq...")
parseq = torch.hub.load('baudm/parseq', 'parseq', pretrained=True).eval().to(device)
parseq_transform = T.Compose([
    T.Resize((32, 128)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
dbnet_detector = PaddleOCR(rec=False, use_angle_cls=False, lang='en', use_gpu=True, show_log=False)

def run_dbnet_parseq(image_rgb: np.ndarray) -> str:
    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    result = dbnet_detector.ocr(image_bgr, rec=False)
    ocr_texts = []
    if result and result[0]:
        h, w = image_rgb.shape[:2]
        for bbox in result[0]:
            pts = np.array(bbox, dtype=np.int32)
            x_min, x_max = max(0, np.min(pts[:, 0]) - 2), min(w, np.max(pts[:, 0]) + 2)
            y_min, y_max = max(0, np.min(pts[:, 1]) - 2), min(h, np.max(pts[:, 1]) + 2)
            if (x_max - x_min) >= 10 and (y_max - y_min) >= 10:
                crop = image_rgb[y_min:y_max, x_min:x_max]
                try:
                    pil_img = Image.fromarray(crop)
                    img_tensor = parseq_transform(pil_img).unsqueeze(0).to(device)
                    with torch.no_grad():
                        logits = parseq(img_tensor).softmax(-1)
                        preds, probs = parseq.tokenizer.decode(logits)
                    if probs[0].mean().item() > 0.5:
                        ocr_texts.append(preds[0])
                except:
                    pass
    return " ".join(ocr_texts)

print("✅ All OCR pipelines loaded")

## 6. Run Full Evaluation

In [ ]:
# Results storage
pipelines = {
    "EasyOCR + TrOCR + Qwen": {"func": run_easyocr_trocr, "results": [], "anls": [], "acc": [], "sem": [], "cer": [], "wer": [], "latencies": []},
    "PaddleOCR + Qwen": {"func": run_paddleocr, "results": [], "anls": [], "acc": [], "sem": [], "cer": [], "wer": [], "latencies": []},
    "DBNet + Parseq + Qwen": {"func": run_dbnet_parseq, "results": [], "anls": [], "acc": [], "sem": [], "cer": [], "wer": [], "latencies": []},
}

def get_image_path(image_id):
    for ext in ['', '.jpg', '.jpeg', '.png']:
        p = Path(DATA_PATH) / f"{image_id}{ext}"
        if p.exists():
            return str(p)
    return None

print(f"\n🚀 Evaluating {LIMIT} samples with blur={BLUR_RADIUS}")
count = 0

for sample in tqdm(samples[:LIMIT*2]):  # Extra samples in case some fail
    if count >= LIMIT:
        break
    
    question = sample['question']
    image_id = sample['image_id']
    gt_answers = sample.get('answers', [])
    
    img_path = get_image_path(image_id)
    if not img_path:
        continue
    
    try:
        # Load and blur image
        img = Image.open(img_path).convert("RGB")
        if BLUR_RADIUS > 0:
            img = img.filter(ImageFilter.GaussianBlur(radius=BLUR_RADIUS))
        temp_path = "/tmp/temp_eval.jpg"
        img.save(temp_path)
        img_array = np.array(img)
        
        for name, pipeline in pipelines.items():
            start = time.time()
            
            # OCR
            ocr_text = pipeline["func"](img_array)
            
            # VQA with Qwen
            pred_answer = ask_qwen(temp_path, question, ocr_text)
            pred_answer = re.sub(r'\s+', ' ', pred_answer).strip()
            
            latency = time.time() - start
            
            # Calculate metrics
            anls = calculate_anls(pred_answer, gt_answers)
            acc = calculate_accuracy(pred_answer, gt_answers)
            sem = judge.judge(question, gt_answers, pred_answer)  # Uses multi-key fallback!
            cer, wer = calculate_cer_wer(pred_answer, gt_answers)
            
            pipeline["anls"].append(anls)
            pipeline["acc"].append(acc)
            pipeline["sem"].append(sem)
            pipeline["cer"].append(cer)
            pipeline["wer"].append(wer)
            pipeline["latencies"].append(latency)
        
        count += 1
        
        # Save checkpoint every 20 samples
        if count % 20 == 0:
            print(f"\n💾 Checkpoint: {count} samples processed")
        
    except RuntimeError as e:
        if "exhausted" in str(e):
            print(f"\n❌ STOPPING: {e}")
            break
        raise
    except Exception as e:
        print(f"Error: {e}")
        continue

print(f"\n✅ Evaluated {count} samples")

## 7. Display Results Table

In [ ]:
print("\n" + "="*100)
print("EVALUATION RESULTS - PRETRAINED BASELINE")
print(f"Blur Level: {BLUR_RADIUS} | Samples: {count}")
print("="*100)
print(f"{'Pipeline':<30} {'Accuracy':<12} {'ANLS':<10} {'Semantic':<12} {'CER':<10} {'WER':<10} {'FPS':<8}")
print("-"*100)

final_results = []

for name, data in pipelines.items():
    acc = np.mean(data["acc"]) * 100 if data["acc"] else 0
    anls = np.mean(data["anls"]) * 100 if data["anls"] else 0
    sem = np.mean(data["sem"]) * 100 if data["sem"] else 0
    cer = np.mean(data["cer"]) * 100 if data["cer"] else 0
    wer = np.mean(data["wer"]) * 100 if data["wer"] else 0
    fps = 1.0 / np.mean(data["latencies"]) if data["latencies"] else 0
    
    print(f"{name:<30} {acc:>10.2f}% {anls:>8.2f}% {sem:>10.2f}% {cer:>8.2f}% {wer:>8.2f}% {fps:>6.2f}")
    
    final_results.append({
        "pipeline": name,
        "accuracy": round(acc, 2),
        "anls": round(anls, 2),
        "semantic_score": round(sem, 2),
        "cer": round(cer, 2),
        "wer": round(wer, 2),
        "fps": round(fps, 2)
    })

print("="*100)

## 8. Save Results

In [ ]:
output = {
    "config": {
        "blur_radius": BLUR_RADIUS,
        "num_samples": count,
        "model_type": "pretrained_baseline"
    },
    "results": final_results
}

with open("pretrained_baseline_results.json", "w") as f:
    json.dump(output, f, indent=2)

print("✅ Results saved to: pretrained_baseline_results.json")
print("\n📥 Download this file to compare with fine-tuned model results!")